# Selieri a- Chess Cheating Detection
## Stockfish vs Maia (LC0): Feature Analysis & RNN Classifier

**Hypothesis:** Stockfish evaluation metrics (CPL, Rank) correlate more strongly with engine-assisted (cheating) moves, while Maia/LC0 metrics correlate more with natural human play. An RNN trained on these per-move sequences should detect cheating at the game level.

**Data:** `features 1.xlsx` (73 SF+LC0 features per game as move sequences) merged with `batch1.xlsx` (per-game labels: cheat/clean, cheater side, per-move binary labels).

---
**Steps:**
1. Upload pre-merged data ()
2. Parse serialized move arrays
3. Data visualisation a- SF vs Maia on cheaters vs non-cheaters
4. RNN (BiLSTM) training & evaluation

In [ ]:
# a-a- 0. Install dependencies a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-
!pip install -q openpyxl seaborn scikit-learn

In [ ]:
# a-a- 1. Imports a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-
import ast, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='deep')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1 - Upload Data
Upload **`combined_features_labels.xlsx`** when prompted (merged batch 1 + batch 2, 2000 games).

In [ ]:
from google.colab import files

print("Upload combined_features_labels.xlsx  (merged batch1 + batch2, 2000 games)")
uploaded = files.upload()

merged_key = list(uploaded.keys())[0]
print(f"Loaded: {merged_key}")

In [ ]:
# - 3. Load merged dataset -
df = pd.read_excel(merged_key)
print(f"Dataset shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
df.head(2)

## 2 a- Parse Serialized Move Arrays
Each feature cell holds a JSON array (one value per move). We explode these into a move-level dataframe for analysis.

In [ ]:
# a-a- 4. Array parser a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-
def parse_array(s):
    """Safely convert a serialised list string to a Python list."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return []
    if isinstance(s, (list, np.ndarray)):
        return list(s)
    s = str(s).strip()
    if not s or s in ('nan', 'None', '[]'):
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        try:
            return json.loads(s)
        except Exception:
            return []

# Columns that hold per-move sequences
ARRAY_COLS = [
    'Side', 'MoveNo', 'UCI', 'SAN', 'FEN_before', 'FEN_after',
    'EMT_ms', 'ClkAfter_s',
    'SF_D10_Rank','SF_D10_CPL','SF_D10_AdvWP','SF_D10_BestWP','SF_D10_WCL',
    'SF_D10_Ambiguity05','SF_D10_difNextBest','SF_D10_difNextWorst','SF_D10_Sharpness',
    'SF_D15_Rank','SF_D15_CPL','SF_D15_AdvWP','SF_D15_BestWP','SF_D15_WCL',
    'SF_D15_Ambiguity05','SF_D15_difNextBest','SF_D15_difNextWorst','SF_D15_Sharpness',
    'SF_D20_Rank','SF_D20_CPL','SF_D20_AdvWP','SF_D20_BestWP','SF_D20_WCL',
    'SF_D20_Ambiguity05','SF_D20_difNextBest','SF_D20_difNextWorst','SF_D20_Sharpness',
    'LC0_D10_Rank','LC0_D10_CPL','LC0_D10_AdvWP','LC0_D10_BestWP','LC0_D10_WCL',
    'LC0_D10_Ambiguity05','LC0_D10_difNextBest','LC0_D10_difNextWorst','LC0_D10_Sharpness',
    'LC0_D15_Rank','LC0_D15_CPL','LC0_D15_AdvWP','LC0_D15_BestWP','LC0_D15_WCL',
    'LC0_D15_Ambiguity05','LC0_D15_difNextBest','LC0_D15_difNextWorst','LC0_D15_Sharpness',
    'LC0_D20_Rank','LC0_D20_CPL','LC0_D20_AdvWP','LC0_D20_BestWP','LC0_D20_WCL',
    'LC0_D20_Ambiguity05','LC0_D20_difNextBest','LC0_D20_difNextWorst','LC0_D20_Sharpness',
    'white_labels', 'black_labels',
]
ARRAY_COLS = [c for c in ARRAY_COLS if c in df.columns]

for col in ARRAY_COLS:
    df[col + '_arr'] = df[col].apply(parse_array)

print('Arrays parsed. Example SF_D20_CPL for game 0:')
print(df['SF_D20_CPL_arr'].iloc[0][:10])

In [ ]:
# a-a- 5. Build move-level dataframe a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-a-
FEAT_COLS_D20 = [
    'SF_D20_Rank','SF_D20_CPL','SF_D20_AdvWP','SF_D20_BestWP','SF_D20_WCL',
    'SF_D20_Ambiguity05','SF_D20_difNextBest','SF_D20_difNextWorst','SF_D20_Sharpness',
    'LC0_D20_Rank','LC0_D20_CPL','LC0_D20_AdvWP','LC0_D20_BestWP','LC0_D20_WCL',
    'LC0_D20_Ambiguity05','LC0_D20_difNextBest','LC0_D20_difNextWorst','LC0_D20_Sharpness',
]

rows = []
for _, game in df.iterrows():
    sides        = game['Side_arr']
    w_labels     = game['white_labels_arr']
    b_labels     = game['black_labels_arr']
    n_moves      = len(sides)
    game_label   = 1 if game['phase'] == 'cheat' else 0

    feat_arrays  = {fc: game[fc + '_arr'] for fc in FEAT_COLS_D20 if fc + '_arr' in game.index}

    w_idx = b_idx = 0
    for i, side in enumerate(sides):
        if side == 'W':
            move_label = int(w_labels[w_idx]) if w_idx < len(w_labels) else 0
            w_idx += 1
        else:
            move_label = int(b_labels[b_idx]) if b_idx < len(b_labels) else 0
            b_idx += 1

        row = {
            'game_id'    : game['game_id'],
            'move_idx'   : i,
            'side'       : side,
            'game_label' : game_label,
            'move_label' : move_label,
            'cheater_side': game['cheater_side'],
        }
        for fc, arr in feat_arrays.items():
            row[fc] = arr[i] if i < len(arr) else np.nan
        rows.append(row)

moves_df = pd.DataFrame(rows)

# Coerce numeric columns
for fc in FEAT_COLS_D20:
    if fc in moves_df.columns:
        moves_df[fc] = pd.to_numeric(moves_df[fc], errors='coerce')

moves_df.dropna(subset=FEAT_COLS_D20, how='all', inplace=True)
moves_df.reset_index(drop=True, inplace=True)

print(f'Move-level dataframe: {moves_df.shape}')
print(f'Cheat moves: {moves_df["move_label"].sum()} / {len(moves_df)}')
moves_df.head(3)

## 3 a- Data Visualisation
### 3.1 Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Phase distribution
phase_counts = df['phase'].value_counts()
axes[0].pie(phase_counts, labels=phase_counts.index, autopct='%1.0f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[0].set_title('Game Phase Distribution')

# Cheater side
cs = df['cheater_side'].value_counts()
axes[1].bar(cs.index, cs.values, color=['#3498db','#e74c3c','#f39c12'])
axes[1].set_title('Cheater Side')
axes[1].set_ylabel('Games')

# Game length distribution
game_lengths = df['SF_D20_CPL_arr'].apply(len)
axes[2].hist(game_lengths, bins=30, color='#9b59b6', edgecolor='white')
axes[2].set_title('Game Length (moves)')
axes[2].set_xlabel('# Moves')
axes[2].set_ylabel('Games')

plt.suptitle('Selieri Dataset a- Batch 1 Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Total games: {len(df)}  |  Total moves analysed: {len(moves_df):,}')

### 3.2 SF vs Maia 2D Quadrant Analysis

The central hypothesis in 2D space: every move lives at a point (SF_Rank, LC0_Rank).
Cheater moves cluster in the **top-left quadrant** - low SF rank (engine best) and high LC0 rank (Maia disagrees).
Human moves cluster in the **bottom-right** - Maia agrees, Stockfish does not find them optimal.
We then split by population to compare Stockfish vs Maia in isolation on each group.

In [ ]:
CPL_CLIP = 300
moves_df['SF_D20_CPL_c']  = moves_df['SF_D20_CPL'].clip(0, CPL_CLIP)
moves_df['LC0_D20_CPL_c'] = moves_df['LC0_D20_CPL'].clip(0, CPL_CLIP)

sf_med  = moves_df['SF_D20_Rank'].median()
lc0_med = moves_df['LC0_D20_Rank'].median()

cheat_moves = moves_df[moves_df['move_label'] == 1]
human_moves = moves_df[moves_df['move_label'] == 0]

sample_cheat = cheat_moves.dropna(subset=['SF_D20_Rank','LC0_D20_Rank']).sample(min(3000, len(cheat_moves)), random_state=42)
sample_human = human_moves.dropna(subset=['SF_D20_Rank','LC0_D20_Rank']).sample(min(3000, len(human_moves)), random_state=42)

fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(sample_human['SF_D20_Rank'], sample_human['LC0_D20_Rank'],
            alpha=0.18, s=8, c='#2ecc71', label='Human move')
ax1.scatter(sample_cheat['SF_D20_Rank'], sample_cheat['LC0_D20_Rank'],
            alpha=0.35, s=8, c='#e74c3c', label='Cheat move')

ax1.axvline(sf_med,  color='white', lw=1.2, ls='--', alpha=0.6)
ax1.axhline(lc0_med, color='white', lw=1.2, ls='--', alpha=0.6)

quadrant_labels = [
    (0.25, 0.78, 'LOW SF / HIGH LC0\n(Cheater Zone)', '#e74c3c'),
    (0.75, 0.78, 'HIGH SF / HIGH LC0\n(Both agree bad)', '#aaaaaa'),
    (0.25, 0.22, 'LOW SF / LOW LC0\n(Both agree good)', '#aaaaaa'),
    (0.75, 0.22, 'HIGH SF / LOW LC0\n(Human Zone)', '#2ecc71'),
]
for (xp, yp, txt, col) in quadrant_labels:
    ax1.text(xp, yp, txt, transform=ax1.transAxes,
             ha='center', va='center', fontsize=8.5, color=col,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#111111', alpha=0.7))

ax1.set_xlabel('Stockfish Rank (1 = engine best move)')
ax1.set_ylabel('Maia/LC0 Rank')
ax1.set_xlim(0.5, 10.5)
ax1.set_ylim(0.5, 10.5)
ax1.set_title('2D Quadrant: SF Rank vs Maia Rank', fontweight='bold')
ax1.legend(markerscale=3, loc='upper right')

ax2 = fig.add_subplot(1, 2, 2)

def quadrant_pct(df):
    sf  = df['SF_D20_Rank']
    lc0 = df['LC0_D20_Rank']
    return [
        ((sf <= sf_med) & (lc0 > lc0_med)).mean(),   # low SF, high LC0 = cheat zone
        ((sf >  sf_med) & (lc0 <= lc0_med)).mean(),  # high SF, low LC0 = human zone
        ((sf <= sf_med) & (lc0 <= lc0_med)).mean(),  # both good
        ((sf >  sf_med) & (lc0 >  lc0_med)).mean(),  # both bad
    ]

q_labels = [
    'Low SF + High LC0\n(Cheater Zone)',
    'High SF + Low LC0\n(Human Zone)',
    'Low SF + Low LC0\n(Both Good)',
    'High SF + High LC0\n(Both Bad)',
]
q_cheat_pct = quadrant_pct(cheat_moves.dropna(subset=['SF_D20_Rank','LC0_D20_Rank']))
q_human_pct = quadrant_pct(human_moves.dropna(subset=['SF_D20_Rank','LC0_D20_Rank']))

x = list(range(len(q_labels)))
w = 0.35
ax2.bar([i - w/2 for i in x], q_cheat_pct, width=w, color='#e74c3c', alpha=0.85, label='Cheat moves')
ax2.bar([i + w/2 for i in x], q_human_pct, width=w, color='#2ecc71', alpha=0.85, label='Human moves')
ax2.set_xticks(x)
ax2.set_xticklabels(q_labels, fontsize=8)
ax2.set_ylabel('Proportion of moves in quadrant')
ax2.set_title('Quadrant Occupancy: Cheat vs Human', fontweight='bold')
ax2.legend()
ax2.set_ylim(0, 0.5)

plt.suptitle('Stockfish vs Maia 2D Analysis -- The Cheater Zone', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Cheat moves in Cheater Zone (low SF, high LC0): {q_cheat_pct[0]:.1%}")
print(f"Human moves in Human Zone   (high SF, low LC0): {q_human_pct[1]:.1%}")


In [ ]:
from scipy.stats import mannwhitneyu

cheat_sf  = cheat_moves['SF_D20_Rank'].dropna().clip(1, 10)
human_sf  = human_moves['SF_D20_Rank'].dropna().clip(1, 10)
cheat_lc0 = cheat_moves['LC0_D20_Rank'].dropna().clip(1, 10)
human_lc0 = human_moves['LC0_D20_Rank'].dropna().clip(1, 10)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# TOP LEFT: Stockfish rank - cheaters vs humans
ax = axes[0][0]
for data, label, color, offset in [(cheat_sf,'Cheat','#e74c3c',0), (human_sf,'Human','#2ecc71',0.4)]:
    counts = data.value_counts().sort_index()
    ax.bar(counts.index + offset, counts / counts.sum(), width=0.4, alpha=0.8, color=color, label=label)
_, p = mannwhitneyu(cheat_sf, human_sf, alternative='less')
ax.set_title(f'Stockfish Rank: Cheat vs Human  (p={p:.2e})', fontweight='bold')
ax.set_xlabel('Rank (1 = engine picks this move)')
ax.set_ylabel('Proportion')
ax.set_xticks(range(1, 11))
ax.legend()
ax.text(0.62, 0.82,
        f'Cheat Rank-1: {(cheat_sf==1).mean():.1%}\nHuman Rank-1: {(human_sf==1).mean():.1%}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#222', alpha=0.8))

# TOP RIGHT: Maia rank - cheaters vs humans
ax = axes[0][1]
for data, label, color, offset in [(cheat_lc0,'Cheat','#e74c3c',0), (human_lc0,'Human','#2ecc71',0.4)]:
    counts = data.value_counts().sort_index()
    ax.bar(counts.index + offset, counts / counts.sum(), width=0.4, alpha=0.8, color=color, label=label)
_, p = mannwhitneyu(human_lc0, cheat_lc0, alternative='less')
ax.set_title(f'Maia/LC0 Rank: Cheat vs Human  (p={p:.2e})', fontweight='bold')
ax.set_xlabel('Rank')
ax.set_ylabel('Proportion')
ax.set_xticks(range(1, 11))
ax.legend()
ax.text(0.62, 0.82,
        f'Cheat Rank-1: {(cheat_lc0==1).mean():.1%}\nHuman Rank-1: {(human_lc0==1).mean():.1%}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#222', alpha=0.8))

# BOTTOM LEFT: CLEAN GAMES ONLY - SF vs Maia CPL
ax = axes[1][0]
clean_game_ids = df[df['phase'] == 'clean']['game_id'].values
clean_moves = moves_df[moves_df['game_id'].isin(clean_game_ids)]
ax.hist(clean_moves['SF_D20_CPL_c'].dropna(),  bins=50, alpha=0.65, color='#e74c3c', density=True, label='Stockfish CPL')
ax.hist(clean_moves['LC0_D20_CPL_c'].dropna(), bins=50, alpha=0.65, color='#3498db', density=True, label='Maia/LC0 CPL')
ax.set_title('CLEAN GAMES ONLY: SF vs Maia CPL', fontweight='bold', color='#2ecc71')
ax.set_xlabel('CPL (clipped 300)')
ax.set_ylabel('Density')
ax.legend()
sf_m = clean_moves['SF_D20_CPL_c'].mean()
lc_m = clean_moves['LC0_D20_CPL_c'].mean()
ax.text(0.52, 0.80,
        f'SF mean CPL:   {sf_m:.1f}\nMaia mean CPL: {lc_m:.1f}\nMaia advantage: {sf_m - lc_m:+.1f}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#222', alpha=0.8))

# BOTTOM RIGHT: CHEAT GAMES ONLY - SF vs Maia CPL
ax = axes[1][1]
cheat_game_ids = df[df['phase'] == 'cheat']['game_id'].values
cheat_game_moves = moves_df[moves_df['game_id'].isin(cheat_game_ids)]
ax.hist(cheat_game_moves['SF_D20_CPL_c'].dropna(),  bins=50, alpha=0.65, color='#e74c3c', density=True, label='Stockfish CPL')
ax.hist(cheat_game_moves['LC0_D20_CPL_c'].dropna(), bins=50, alpha=0.65, color='#3498db', density=True, label='Maia/LC0 CPL')
ax.set_title('CHEAT GAMES ONLY: SF vs Maia CPL', fontweight='bold', color='#e74c3c')
ax.set_xlabel('CPL (clipped 300)')
ax.set_ylabel('Density')
ax.legend()
sf_m = cheat_game_moves['SF_D20_CPL_c'].mean()
lc_m = cheat_game_moves['LC0_D20_CPL_c'].mean()
ax.text(0.52, 0.80,
        f'SF mean CPL:   {sf_m:.1f}\nMaia mean CPL: {lc_m:.1f}\nSF advantage: {lc_m - sf_m:+.1f}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#222', alpha=0.8))

plt.suptitle('Engine-vs-Engine Comparison Split by Population', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey takeaway:")
print("  Clean games -> Maia should have LOWER CPL (agrees more with human moves)")
print("  Cheat games -> Stockfish should have LOWER CPL (agrees more with engine moves)")


### 3.3 Scatter: SF CPL vs Maia CPL a- coloured by move label

In [ ]:
sample = moves_df.dropna(subset=['SF_D20_CPL_c','LC0_D20_CPL_c']).sample(min(8000, len(moves_df)), random_state=SEED)

fig, ax = plt.subplots(figsize=(8, 7))
for lbl, color in palette.items():
    sub = sample[sample['move_label']==lbl]
    ax.scatter(sub['SF_D20_CPL_c'], sub['LC0_D20_CPL_c'],
               alpha=0.25, s=6, c=color, label=labels[lbl])

ax.set_xlabel('Stockfish CPL (clipped 300)')
ax.set_ylabel('Maia/LC0 CPL (clipped 300)')
ax.set_title('SF CPL vs Maia CPL per Move', fontweight='bold')
ax.legend(markerscale=4)

# Diagonal: SF=LC0 line
lims = [0, CPL_CLIP]
ax.plot(lims, lims, 'k--', alpha=0.3, label='SF = LC0')
ax.set_xlim(lims); ax.set_ylim(lims)
plt.tight_layout()
plt.show()

### 3.4 Violin plots a- all D20 metrics, cheat vs clean at game level

In [ ]:
# Aggregate to game level (mean per metric)
game_agg = moves_df.groupby('game_id')[FEAT_COLS_D20].mean()
game_agg = game_agg.join(df.set_index('game_id')[['phase']])
game_agg['phase_label'] = (game_agg['phase'] == 'cheat').astype(int)

# Key metrics to highlight
KEY_PAIRS = [
    ('SF_D20_CPL',  'LC0_D20_CPL'),
    ('SF_D20_Rank', 'LC0_D20_Rank'),
    ('SF_D20_WCL',  'LC0_D20_WCL'),
    ('SF_D20_Sharpness', 'LC0_D20_Sharpness'),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for idx, (sf_col, lc_col) in enumerate(KEY_PAIRS):
    for jdx, (col, engine) in enumerate([(sf_col,'Stockfish'), (lc_col,'Maia/LC0')]):
        ax = axes[idx*2 + jdx]
        plot_df = pd.melt(
            game_agg[['phase', col]].rename(columns={col:'value'}),
            id_vars='phase', value_vars='value'
        )
        sns.violinplot(data=game_agg, x='phase', y=col, ax=ax,
                       palette={'cheat':'#e74c3c','clean':'#2ecc71'},
                       inner='quartile', cut=0)
        metric_name = col.replace('SF_D20_','').replace('LC0_D20_','')
        ax.set_title(f'{engine}\n{metric_name}', fontsize=10, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')

plt.suptitle('Game-Level Mean Feature Distributions a- Cheat vs Clean', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Correlation Heatmap a- SF vs LC0 features

In [ ]:
corr = game_agg[FEAT_COLS_D20 + ['phase_label']].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', annot_kws={'size':6},
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix a- SF & LC0 D20 Features + Cheat Label', fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Hypothesis Summary a- Effect Sizes

In [ ]:
from scipy.stats import mannwhitneyu

results = []
for col in FEAT_COLS_D20:
    if col not in game_agg.columns:
        continue
    cheat = game_agg[game_agg['phase']=='cheat'][col].dropna()
    clean = game_agg[game_agg['phase']=='clean'][col].dropna()
    if len(cheat) < 5 or len(clean) < 5:
        continue
    _, p = mannwhitneyu(cheat, clean, alternative='two-sided')
    # Cohen's d
    pooled_std = np.sqrt((cheat.std()**2 + clean.std()**2) / 2)
    d = (cheat.mean() - clean.mean()) / (pooled_std + 1e-9)
    results.append({'feature': col, 'cheat_mean': cheat.mean(),
                    'clean_mean': clean.mean(), 'cohens_d': d, 'p_value': p})

res_df = pd.DataFrame(results).sort_values('cohens_d', key=abs, ascending=False)
print(res_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#e74c3c' if 'SF_' in f else '#3498db' for f in res_df['feature']]
ax.barh(res_df['feature'], res_df['cohens_d'], color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel("Cohen's d  (cheat a' clean)")
ax.set_title('Feature Discriminability: Cheat vs Clean Games', fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c',label='Stockfish'),
                   Patch(color='#3498db',label='Maia/LC0')])
plt.tight_layout()
plt.show()

## 4 — BiLSTM: Per-Move Cheating Detection
### 4.1 Build Sequences with Per-Move Labels

Each game becomes a sequence of shape `(n_moves, 18)`. The target `label_sequences` is a matching `(n_moves,)` array of 0/1 — 1 means that specific move was engine-assisted.

In [ ]:
FEATURE_NAMES = FEAT_COLS_D20  # 18 features: 9 SF_D20 + 9 LC0_D20

sequences, label_sequences = [], []

for gid, group in sorted(moves_df.groupby('game_id'), key=lambda x: x[0]):
    seq = group[FEATURE_NAMES].values.astype(np.float32)
    lbl = group['move_label'].values.astype(np.float32)
    if seq.shape[0] < 5:
        continue
    sequences.append(seq)
    label_sequences.append(lbl)

lengths = [s.shape[0] for s in sequences]
total_moves = sum(lengths)
total_cheat = int(sum(l.sum() for l in label_sequences))

print(f'Games            : {len(sequences)}')
print(f'Move counts      : min={min(lengths)}  max={max(lengths)}  mean={np.mean(lengths):.1f}')
print(f'Total moves      : {total_moves:,}')
print(f'Cheat moves      : {total_cheat:,} ({total_cheat/total_moves:.1%})')
print(f'Class imbalance  : {(total_moves-total_cheat)/max(total_cheat,1):.1f}:1  (clean:cheat)')

In [ ]:
# Normalise features across all training moves
all_moves_arr = np.concatenate(sequences, axis=0)
scaler = StandardScaler()
scaler.fit(np.nan_to_num(all_moves_arr, nan=0.0))

sequences_norm = [
    scaler.transform(np.nan_to_num(s, nan=0.0)).astype(np.float32)
    for s in sequences
]

# Train / Val / Test split (70/15/15)
# Stratify by whether any move in the game was a cheat move
game_has_cheat = np.array([int(l.sum() > 0) for l in label_sequences])
idx = np.arange(len(sequences_norm))

tr_idx, tmp_idx, _, y_tmp = train_test_split(idx, game_has_cheat, test_size=0.30, stratify=game_has_cheat, random_state=SEED)
va_idx, te_idx, _, _      = train_test_split(tmp_idx, y_tmp,      test_size=0.50, stratify=y_tmp,          random_state=SEED)

# Compute pos_weight from training set to handle class imbalance
tr_moves_flat = np.concatenate([label_sequences[i] for i in tr_idx])
n_neg = (tr_moves_flat == 0).sum()
n_pos = (tr_moves_flat == 1).sum()
pos_weight_val = n_neg / max(n_pos, 1)

print(f'Train: {len(tr_idx)}  Val: {len(va_idx)}  Test: {len(te_idx)}')
print(f'pos_weight = {pos_weight_val:.2f}  (upweights cheat moves to correct imbalance)')

### 4.2 Dataset & DataLoader

In [ ]:
def collate_fn(batch):
    seqs, lbls = zip(*batch)
    lengths  = torch.tensor([s.shape[0] for s in seqs], dtype=torch.long)
    max_len  = lengths.max().item()
    n_feat   = seqs[0].shape[1]
    padded_x = torch.zeros(len(seqs), max_len, n_feat)
    padded_y = torch.zeros(len(seqs), max_len)          # pad labels with 0 (clean)
    for i, (s, l, length) in enumerate(zip(seqs, lbls, lengths)):
        padded_x[i, :length] = torch.tensor(s)
        padded_y[i, :length] = torch.tensor(l)
    return padded_x, lengths, padded_y


class ChessSeqDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        return sequences_norm[idx], label_sequences[idx]


BATCH = 32
tr_loader = DataLoader(ChessSeqDataset(tr_idx), batch_size=BATCH, shuffle=True,  collate_fn=collate_fn)
va_loader = DataLoader(ChessSeqDataset(va_idx), batch_size=BATCH, shuffle=False, collate_fn=collate_fn)
te_loader = DataLoader(ChessSeqDataset(te_idx), batch_size=BATCH, shuffle=False, collate_fn=collate_fn)
print('DataLoaders ready')

### 4.3 Per-Move BiLSTM Model

Instead of collapsing to one output via attention pooling, the model now outputs a cheat logit **for every timestep** — shape `(B, T)`. Padding positions are masked out of the loss.

In [ ]:
class CheatDetectorPerMove(nn.Module):
    """
    Bidirectional LSTM that outputs a cheat probability for every move.
    Input : (B, T, n_features)
    Output: (B, T) logits  — one per move, padding ignored during loss.
    """
    def __init__(self, n_features=18, hidden=128, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)  # (B, T, 2H)
        logits = self.classifier(out).squeeze(-1)            # (B, T)
        return logits


model = CheatDetectorPerMove(n_features=len(FEATURE_NAMES)).to(DEVICE)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

### 4.4 Training

In [ ]:
EPOCHS   = 40
LR       = 1e-3
PATIENCE = 7

pos_weight_tensor = torch.tensor([pos_weight_val], device=DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor, reduction='none')
optimizer  = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
best_val_loss = float('inf')
patience_ctr  = 0
best_state    = None


def run_epoch(loader, train=True):
    model.train(train)
    total_loss, total_batches = 0.0, 0
    all_preds, all_true = [], []

    with torch.set_grad_enabled(train):
        for x, lengths, y in loader:
            x, lengths, y = x.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)
            logits = model(x, lengths)                                    # (B, T)

            # Build mask: True for real moves, False for padding
            mask = torch.arange(logits.size(1), device=DEVICE)[None, :] < lengths[:, None]

            loss = criterion(logits, y)                                   # (B, T)
            loss = (loss * mask.float()).sum() / mask.float().sum()       # mean over real moves

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            # Collect unpadded predictions for F1
            probs = torch.sigmoid(logits).detach()
            all_preds.extend((probs[mask] >= 0.5).long().cpu().numpy())
            all_true.extend(y[mask].long().cpu().numpy())

            total_loss   += loss.item()
            total_batches += 1

    epoch_f1 = f1_score(all_true, all_preds, zero_division=0)
    return total_loss / total_batches, epoch_f1


print(f'{"Epoch":>5} {"Tr Loss":>9} {"Va Loss":>9} {"Tr F1":>8} {"Va F1":>8}')
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(tr_loader, train=True)
    va_loss, va_f1 = run_epoch(va_loader, train=False)
    scheduler.step(va_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_f1'].append(tr_f1)
    history['val_f1'].append(va_f1)

    print(f'{epoch:5d} {tr_loss:9.4f} {va_loss:9.4f} {tr_f1:8.3f} {va_f1:8.3f}')

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_state)
print('Best model restored.')

### 4.5 Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history['train_loss'], label='Train', color='#3498db')
ax1.plot(history['val_loss'],   label='Val',   color='#e74c3c')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history['train_f1'], label='Train', color='#3498db')
ax2.plot(history['val_f1'],   label='Val',   color='#e74c3c')
ax2.set_title('F1 Score (per-move)'); ax2.set_xlabel('Epoch')
ax2.set_ylim(0, 1.0); ax2.legend()

plt.suptitle('Training Curves — Per-Move BiLSTM', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.6 Test Evaluation — Per-Move + Game-Level

In [ ]:
model.eval()
all_probs, all_preds, all_true = [], [], []
game_probs_list, game_true_list = [], []

with torch.no_grad():
    for x, lengths, y in te_loader:
        x, lengths = x.to(DEVICE), lengths.to(DEVICE)
        logits = model(x, lengths)
        probs  = torch.sigmoid(logits).cpu().numpy()

        for i in range(len(lengths)):
            L = lengths[i].item()
            p = probs[i, :L]
            t = y[i, :L].numpy().astype(int)

            all_probs.extend(p)
            all_preds.extend((p >= 0.5).astype(int))
            all_true.extend(t)

            # Game-level: flag game if any move exceeds threshold
            game_probs_list.append(float(p.max()))
            game_true_list.append(int(t.max()))

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_true   = np.array(all_true)
game_probs = np.array(game_probs_list)
game_true  = np.array(game_true_list)
game_preds = (game_probs >= 0.5).astype(int)

# Per-move metrics
move_acc  = accuracy_score(all_true, all_preds)
move_prec = precision_score(all_true, all_preds, zero_division=0)
move_rec  = recall_score(all_true, all_preds, zero_division=0)
move_f1   = f1_score(all_true, all_preds, zero_division=0)
move_auc  = roc_auc_score(all_true, all_probs)

# Game-level metrics
game_acc = accuracy_score(game_true, game_preds)
game_f1  = f1_score(game_true, game_preds, zero_division=0)
game_auc = roc_auc_score(game_true, game_probs)

full_results = {
    'name': 'Full Model (SF + Maia)',
    'probs': all_probs, 'preds': all_preds, 'true': all_true,
    'game_probs': game_probs, 'game_preds': game_preds, 'game_true': game_true,
    'move_acc': move_acc, 'move_prec': move_prec, 'move_rec': move_rec,
    'move_f1': move_f1, 'move_auc': move_auc,
    'game_acc': game_acc, 'game_f1': game_f1, 'game_auc': game_auc,
    'history': history,
}

print('=' * 50)
print('  PER-MOVE METRICS')
print(f'  Accuracy  : {move_acc:.4f}')
print(f'  Precision : {move_prec:.4f}')
print(f'  Recall    : {move_rec:.4f}')
print(f'  F1        : {move_f1:.4f}')
print(f'  ROC-AUC   : {move_auc:.4f}')
print()
print('  GAME-LEVEL METRICS  (any cheat move -> game flagged)')
print(f'  Accuracy  : {game_acc:.4f}')
print(f'  F1        : {game_f1:.4f}')
print(f'  ROC-AUC   : {game_auc:.4f}')
print('=' * 50)
print()
print(classification_report(all_true, all_preds, target_names=['Clean Move', 'Cheat Move']))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Per-move confusion matrix
cm = confusion_matrix(all_true, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Clean', 'Cheat'], yticklabels=['Clean', 'Cheat'])
axes[0, 0].set_title('Per-Move Confusion Matrix', fontweight='bold')
axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('Actual')

# ROC curves (move-level and game-level)
fpr_m, tpr_m, _ = roc_curve(all_true, all_probs)
fpr_g, tpr_g, _ = roc_curve(game_true, game_probs)
axes[0, 1].plot(fpr_m, tpr_m, color='#3498db', lw=2, label=f'Per-Move  AUC={move_auc:.3f}')
axes[0, 1].plot(fpr_g, tpr_g, color='#e74c3c', lw=2, ls='--', label=f'Game-Level AUC={game_auc:.3f}')
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0, 1].set_xlabel('False Positive Rate'); axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curves', fontweight='bold'); axes[0, 1].legend()

# Predicted probability distribution
axes[1, 0].hist(all_probs[all_true == 0], bins=50, alpha=0.6, color='#2ecc71', density=True, label='Clean moves')
axes[1, 0].hist(all_probs[all_true == 1], bins=50, alpha=0.6, color='#e74c3c', density=True, label='Cheat moves')
axes[1, 0].set_xlabel('Predicted Cheat Probability')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Score Distribution by True Label', fontweight='bold')
axes[1, 0].legend()

# Game-level confusion matrix
cm_g = confusion_matrix(game_true, game_preds)
sns.heatmap(cm_g, annot=True, fmt='d', cmap='Oranges', ax=axes[1, 1],
            xticklabels=['Clean', 'Cheat'], yticklabels=['Clean', 'Cheat'])
axes[1, 1].set_title('Game-Level Confusion Matrix\n(any predicted cheat move → game flagged)', fontweight='bold')
axes[1, 1].set_xlabel('Predicted'); axes[1, 1].set_ylabel('Actual')

plt.suptitle(
    f'Per-Move BiLSTM  |  Move F1={move_f1:.3f}  Move AUC={move_auc:.3f}  Game AUC={game_auc:.3f}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

### 4.7 Attention a- Which Moves Does the Model Focus On?

In [ ]:
# Extract attention weights for a sample of cheat games
model.eval()
attn_records = []

cheat_game_ids = [te_idx[i] for i, lbl in enumerate(all_true) if lbl == 1][:20]

for gid in cheat_game_ids:
    seq = torch.tensor(sequences_norm[gid]).unsqueeze(0).to(DEVICE)  # (1, T, F)
    L   = torch.tensor([seq.shape[1]]).to(DEVICE)
    with torch.no_grad():
        packed = pack_padded_sequence(seq, L.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = model.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        w = torch.softmax(model.attn(out).squeeze(-1), dim=1).cpu().squeeze().numpy()
    n = len(w)
    # normalise position to 0-1
    positions = np.arange(n) / max(n - 1, 1)
    attn_records.append({'positions': positions, 'weights': w})

# Plot mean attention vs move position (relative)
bins = np.linspace(0, 1, 21)
bin_means = np.zeros(20)
bin_cnts  = np.zeros(20)
for rec in attn_records:
    b_idx = np.digitize(rec['positions'], bins) - 1
    b_idx = np.clip(b_idx, 0, 19)
    for bi, w in zip(b_idx, rec['weights']):
        bin_means[bi] += w
        bin_cnts[bi]  += 1
bin_means /= np.maximum(bin_cnts, 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(bins[:-1], bin_means, width=0.04, color='#e74c3c', alpha=0.8, align='edge')
ax.set_xlabel('Relative Move Position in Game (0=opening, 1=endgame)')
ax.set_ylabel('Mean Attention Weight')
ax.set_title('Where Does the RNN Focus? (Cheat Games)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5 - Control: Stockfish-Only Model

To isolate the contribution of Maia/LC0 features, we train an identical BiLSTM using **only the 9 Stockfish D20 features** (no Maia data). The two models are then compared directly - if Maia adds signal, the full model should outperform the SF-only control.

In [ ]:
SF_ONLY_FEATURES = [
    'SF_D20_Rank', 'SF_D20_CPL', 'SF_D20_AdvWP', 'SF_D20_BestWP', 'SF_D20_WCL',
    'SF_D20_Ambiguity05', 'SF_D20_difNextBest', 'SF_D20_difNextWorst', 'SF_D20_Sharpness',
]

sf_sequences = [
    group[SF_ONLY_FEATURES].values.astype(np.float32)
    for _, group in sorted(moves_df.groupby('game_id'), key=lambda x: x[0])
    if group.shape[0] >= 5
]

sf_all = np.concatenate(sf_sequences, axis=0)
sf_scaler = StandardScaler()
sf_scaler.fit(np.nan_to_num(sf_all, nan=0.0))
sf_sequences_norm = [sf_scaler.transform(np.nan_to_num(s, nan=0.0)).astype(np.float32) for s in sf_sequences]


class SFDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        return sf_sequences_norm[idx], label_sequences[idx]


sf_tr_loader = DataLoader(SFDataset(tr_idx), batch_size=BATCH, shuffle=True,  collate_fn=collate_fn)
sf_va_loader = DataLoader(SFDataset(va_idx), batch_size=BATCH, shuffle=False, collate_fn=collate_fn)
sf_te_loader = DataLoader(SFDataset(te_idx), batch_size=BATCH, shuffle=False, collate_fn=collate_fn)

sf_model     = CheatDetectorPerMove(n_features=len(SF_ONLY_FEATURES)).to(DEVICE)
sf_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor, reduction='none')
sf_optimizer = optim.Adam(sf_model.parameters(), lr=LR, weight_decay=1e-4)
sf_scheduler = optim.lr_scheduler.ReduceLROnPlateau(sf_optimizer, patience=3, factor=0.5)

sf_history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
sf_best_val, sf_patience, sf_best_state = float('inf'), 0, None


def run_sf_epoch(loader, train=True):
    sf_model.train(train)
    total_loss, total_batches = 0.0, 0
    all_preds, all_true = [], []
    with torch.set_grad_enabled(train):
        for x, lengths, y in loader:
            x, lengths, y = x.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)
            logits = sf_model(x, lengths)
            mask   = torch.arange(logits.size(1), device=DEVICE)[None, :] < lengths[:, None]
            loss   = sf_criterion(logits, y)
            loss   = (loss * mask.float()).sum() / mask.float().sum()
            if train:
                sf_optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(sf_model.parameters(), 1.0); sf_optimizer.step()
            probs = torch.sigmoid(logits).detach()
            all_preds.extend((probs[mask] >= 0.5).long().cpu().numpy())
            all_true.extend(y[mask].long().cpu().numpy())
            total_loss += loss.item(); total_batches += 1
    return total_loss / total_batches, f1_score(all_true, all_preds, zero_division=0)


print('Training SF-only control model...')
print(f'{"Epoch":>5} {"Tr Loss":>9} {"Va Loss":>9} {"Tr F1":>8} {"Va F1":>8}')
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1 = run_sf_epoch(sf_tr_loader, True)
    va_loss, va_f1 = run_sf_epoch(sf_va_loader, False)
    sf_scheduler.step(va_loss)
    sf_history['train_loss'].append(tr_loss); sf_history['val_loss'].append(va_loss)
    sf_history['train_f1'].append(tr_f1);     sf_history['val_f1'].append(va_f1)
    print(f'{epoch:5d} {tr_loss:9.4f} {va_loss:9.4f} {tr_f1:8.3f} {va_f1:8.3f}')
    if va_loss < sf_best_val:
        sf_best_val   = va_loss
        sf_best_state = {k: v.cpu().clone() for k, v in sf_model.state_dict().items()}
        sf_patience   = 0
    else:
        sf_patience += 1
        if sf_patience >= PATIENCE:
            print(f'Early stopping at epoch {epoch}'); break

sf_model.load_state_dict(sf_best_state)

# Evaluate
sf_model.eval()
sf_probs, sf_preds, sf_true = [], [], []
sf_game_probs, sf_game_true = [], []
with torch.no_grad():
    for x, lengths, y in sf_te_loader:
        x, lengths = x.to(DEVICE), lengths.to(DEVICE)
        logits = sf_model(x, lengths)
        probs  = torch.sigmoid(logits).cpu().numpy()
        for i in range(len(lengths)):
            L = lengths[i].item()
            p = probs[i, :L]; t = y[i, :L].numpy().astype(int)
            sf_probs.extend(p); sf_preds.extend((p >= 0.5).astype(int)); sf_true.extend(t)
            sf_game_probs.append(float(p.max())); sf_game_true.append(int(t.max()))

sf_probs = np.array(sf_probs); sf_preds = np.array(sf_preds); sf_true = np.array(sf_true)
sf_game_probs = np.array(sf_game_probs); sf_game_true = np.array(sf_game_true)

sf_results = {
    'name': 'Control (SF only)',
    'probs': sf_probs, 'preds': sf_preds, 'true': sf_true,
    'game_probs': sf_game_probs,
    'move_acc':  accuracy_score(sf_true, sf_preds),
    'move_prec': precision_score(sf_true, sf_preds, zero_division=0),
    'move_rec':  recall_score(sf_true, sf_preds, zero_division=0),
    'move_f1':   f1_score(sf_true, sf_preds, zero_division=0),
    'move_auc':  roc_auc_score(sf_true, sf_probs),
    'game_auc':  roc_auc_score(sf_game_true, sf_game_probs),
    'history': sf_history,
}
print(f'\nSF-only  move F1={sf_results["move_f1"]:.4f}  move AUC={sf_results["move_auc"]:.4f}  game AUC={sf_results["game_auc"]:.4f}')

In [ ]:
models_list  = [full_results, sf_results]
colors_list  = ['#3498db', '#e74c3c']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Metric bar chart
met_keys  = ['move_acc', 'move_prec', 'move_rec', 'move_f1', 'move_auc']
met_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
x = np.arange(len(met_keys)); w = 0.35
ax = axes[0, 0]
for i, (res, col) in enumerate(zip(models_list, colors_list)):
    vals = [res[m] for m in met_keys]
    bars = ax.bar(x + i * w, vals, width=w, color=col, alpha=0.85, label=res['name'])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)
ax.set_xticks(x + w / 2); ax.set_xticklabels(met_names, fontsize=9)
ax.set_ylim(0, 1.12); ax.set_title('Per-Move Metrics', fontweight='bold'); ax.legend(fontsize=8)

# ROC curve
ax = axes[0, 1]
for res, col in zip(models_list, colors_list):
    fpr, tpr, _ = roc_curve(res['true'], res['probs'])
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{res["name"]}  AUC={res["move_auc"]:.3f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_title('ROC — Per-Move', fontweight='bold'); ax.legend(fontsize=8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')

# Val loss
ax = axes[0, 2]
for res, col in zip(models_list, colors_list):
    ax.plot(res['history']['val_loss'], color=col, lw=2, label=res['name'])
    ax.plot(res['history']['val_f1'],   color=col, lw=2, ls='--', alpha=0.6)
ax.set_title('Val Loss (solid) & Val F1 (dashed)', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlabel('Epoch')

# Confusion matrices
for idx, (res, cmap) in enumerate(zip(models_list, ['Blues', 'Reds'])):
    ax = axes[1, idx]
    cm = confusion_matrix(res['true'], res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Clean', 'Cheat'], yticklabels=['Clean', 'Cheat'])
    ax.set_title(f'{res["name"]}\nMove F1={res["move_f1"]:.3f}  AUC={res["move_auc"]:.3f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# Delta summary
ax = axes[1, 2]; ax.axis('off')
lines = ['Maia contribution (Full — SF only):\n\n']
for k, n in zip(met_keys, met_names):
    d = full_results[k] - sf_results[k]
    lines.append(f'  {n:<12}: {d:+.4f}\n')
lines += [f'\n  Game AUC delta: {full_results["game_auc"] - sf_results["game_auc"]:+.4f}\n']
lines += ['\nPositive = Maia features help\nNegative = SF alone was better']
ax.text(0.05, 0.95, ''.join(lines), transform=ax.transAxes, fontsize=11, va='top',
        family='monospace', bbox=dict(boxstyle='round', facecolor='#1a1a2e', alpha=0.8))

plt.suptitle('Full (SF + Maia) vs Control (SF only) — Per-Move Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

## 6 — Results Summary

| Metric | Full (SF + Maia) | Control (SF only) | Maia Δ |
|--------|-----------------|-------------------|--------|
| Move Accuracy | see output | see output | see output |
| Move Precision | see output | see output | see output |
| Move Recall | see output | see output | see output |
| Move F1 | see output | see output | see output |
| Move AUC | see output | see output | see output |
| Game AUC | see output | see output | see output |

**What this means:**
- **Per-move F1 / AUC**: how well the model identifies the specific moves that were engine-assisted
- **Game AUC**: derived by taking `max(move_prob)` per game — still useful for flagging games
- **Maia Δ**: if positive, Maia/LC0 features add discriminative signal beyond Stockfish alone

In [ ]:
df.to_csv('combined_features_labels.csv', index=False)
files.download('combined_features_labels.csv')
print('Downloaded combined_features_labels.csv')